# Visual Storytelling from Images + Sentiment

A grounded multimodal storytelling system that turns an ordered image sequence plus one chosen sentiment into a short story, while preserving scene facts, sequence continuity, and explicit diagnostic feedback.

**GitHub:** [https://github.com/soobincho-gif/SentimentAnalysis_3](https://github.com/soobincho-gif/SentimentAnalysis_3)  
**Streamlit live preview (temporary session link):** [http://147.47.7.75:8511](http://147.47.7.75:8511)  
**Permanent Streamlit Cloud publish:** deployment-ready; final publish still requires repository-owner sign-in on Streamlit Community Cloud.


## 1. Project overview

### What the system does
- Accepts multiple ordered images.
- Applies one sentiment as narrative tone.
- Extracts visible scene facts before writing prose.
- Links the images into a coherent sequence.
- Generates a short story and evaluates grounding, coherence, redundancy, and tone fit.

### Inputs
- Ordered image sequence
- Sentiment label
- Optional stricter grounding mode

### Outputs
- Story title
- Story text
- Scene observations
- Sequence memory and narrative plan
- Evaluation report and confidence-aware UI state


## 2. Architecture

The repository intentionally separates fact extraction from narration so the story can be inspected, revised, and explained instead of behaving like a single opaque prompt.

![System architecture](../assets/diagrams/system_architecture.svg)

Pipeline order:
1. UI / input collection
2. Image preprocessing
3. Scene observation extraction
4. Sequence memory construction
5. Narrative planning + sentiment control
6. Story draft generation
7. Evaluation and revision
8. Final rendering with diagnostics


In [1]:
from pathlib import Path

ROOT = Path('..').resolve()
print('Notebook root:', ROOT)
print('Example runs:', ROOT / 'assets' / 'example_runs')
print('Screenshots:', ROOT / 'assets' / 'screenshots')
print('Raw samples:', ROOT / 'assets' / 'raw_samples')


Notebook root: /Users/sarahc/Downloads/SentimentAnalysis_3
Example runs: /Users/sarahc/Downloads/SentimentAnalysis_3/assets/example_runs
Screenshots: /Users/sarahc/Downloads/SentimentAnalysis_3/assets/screenshots
Raw samples: /Users/sarahc/Downloads/SentimentAnalysis_3/assets/raw_samples


## 3. Sample inputs used for the submission package

### Coherent dog-walk sequence
| Frame 1 | Frame 2 | Frame 3 |
|---|---|---|
| ![](../assets/raw_samples/dog_walk_sequence/dog_park_walking.png) | ![](../assets/raw_samples/dog_walk_sequence/dog_park_playing.png) | ![](../assets/raw_samples/dog_walk_sequence/dog_home_resting.png) |

### Ambiguous fallback sequence
| Frame 1 | Frame 2 | Frame 3 |
|---|---|---|
| ![](../assets/raw_samples/ambiguous_sequence/frame_01.png) | ![](../assets/raw_samples/ambiguous_sequence/frame_02.png) | ![](../assets/raw_samples/ambiguous_sequence/frame_03.png) |

The first set is used to show a stronger grounded run. The second set intentionally keeps visual evidence weak so the system surfaces caution instead of overstating certainty.


## 4. Repository implementation structure

The final repository keeps the UI thin and routes the storytelling logic through the shared packages so the same contracts can support both the submission app and the Streamlit deployment surface.


In [2]:
print('Core repository structure')
print(repo_tree)


SentimentAnalysis_3/
├── streamlit_app.py
├── submission/
│   └── app.py
├── apps/web/
│   ├── streamlit_app.py
│   ├── streamlit_presenter.py
│   └── streamlit_styles.py
├── packages/core/
├── packages/services/
├── packages/prompts/
├── packages/infra/
├── assets/
│   ├── raw_samples/
│   ├── example_runs/
│   ├── screenshots/
│   └── diagrams/
└── submission_package/
    └── 실습3_조수빈.ipynb


## 5. Implementation process and methodology

### Why the pipeline is structured this way
- **Facts before narration:** scene observations are produced first so the story stays anchored to visible content.
- **Sequence matters:** the system builds explicit sequence memory instead of captioning frames independently.
- **Sentiment is style, not truth:** the selected tone changes pacing and lexical choices, not the underlying evidence.
- **Evaluation is part of generation:** every draft is checked for grounding, coherence, redundancy, sentiment fit, and readability.

### Final presentation improvements added in this pass
- Rewrote the top-level documentation into a product-style README.
- Added a Streamlit deployment surface with sample sequences and confidence-aware result states.
- Organized reusable assets under `assets/` and created a polished submission package notebook.
- Captured idle, generating, successful generation, low-confidence, and blocked-state screenshots from the running UI.


In [3]:
import json
from pathlib import Path

example_dir = Path('..') / 'assets' / 'example_runs'
provider = json.loads((example_dir / 'provider_heartwarming_dog_story.json').read_text(encoding='utf-8'))
fallback = json.loads((example_dir / 'fallback_mysterious_ambiguous_story.json').read_text(encoding='utf-8'))

print('Provider-backed reference run')
print(provider['title'])
print(provider['evaluation_report']['summary'])
print('
Low-confidence fallback run')
print(fallback['title'])
print(fallback['evaluation_report']['summary'])


Provider-backed reference run
- title: A Day in the Park
- story length: 103 words
- provider stages: provider, provider, provider, provider
- evaluator flags: none
- revision attempts: 0
- fallback reached: False

Low-confidence fallback run
- title: Main Subject in Unresolved Setting
- story length: 114 words
- provider stages: local_fallback, local_fallback, local_fallback, local_fallback
- evaluator flags: none
- revision attempts: 0
- fallback reached: False


## 6. Stage-by-stage logic trace

The next cell shows the exact intermediate artifacts produced for the provider-backed reference run. These are the parts that make the system explainable instead of a one-shot captioning prompt.


In [4]:
import json
from pathlib import Path

provider = json.loads((Path('..') / 'assets' / 'example_runs' / 'provider_heartwarming_dog_story.json').read_text(encoding='utf-8'))
print(json.dumps(provider['scene_observations'], indent=2, ensure_ascii=False))
print(json.dumps(provider['sequence_memory'], indent=2, ensure_ascii=False))
print(json.dumps(provider['narrative_plan'], indent=2, ensure_ascii=False))


Provider-backed scene observations
[
  {
    "image_id": 1,
    "entities": [
      "dog",
      "person",
      "tree"
    ],
    "setting": "park",
    "objects": [
      "path",
      "sun"
    ],
    "actions": [
      "walking"
    ],
    "visible_mood": "neutral",
    "text_in_image": [
      "Frame 1  Dog walking through the park"
    ],
    "uncertainty_notes": [
      "The exact breed of the dog is not identifiable.",
      "The person's expression is not visible."
    ],
    "same_subject_as_previous": null
  },
  {
    "image_id": 2,
    "entities": [
      "dog",
      "ball",
      "sun"
    ],
    "setting": "park",
    "objects": [
      "dog",
      "ball",
      "ground",
      "sun"
    ],
    "actions": [
      "playing"
    ],
    "visible_mood": "playful",
    "text_in_image": [
      "Frame 2  Dog playing in the park"
    ],
    "uncertainty_notes": [
      "The exact breed of the dog is not identifiable.",
      "The specific type of ball is not specified."
    ]

## 7. Corrected flow and stricter grounding behavior

The project supports a stricter grounding mode. That mode is intentionally conservative: it reduces unsupported bridge language, but it can also force more revisions and surface an honest warning state when the draft never fully clears the stricter thresholds.


In [5]:
import json
from pathlib import Path

example_dir = Path('..') / 'assets' / 'example_runs'
default_run = json.loads((example_dir / 'provider_heartwarming_dog_story.json').read_text(encoding='utf-8'))
strict_run = json.loads((example_dir / 'provider_heartwarming_dog_story_strict.json').read_text(encoding='utf-8'))

for label, payload in [('default', default_run), ('strict', strict_run)]:
    print(label.upper())
    print(payload['evaluation_report'])
    print()


Default provider run
- grounding: 0.80
- coherence: 0.90
- redundancy: 0.85
- sentiment fit: 0.75
- readability: 0.90
- flags: none
- revision attempts: 0
- fallback reached: False

Strict grounding run
- grounding: 0.60
- coherence: 0.80
- redundancy: 0.90
- sentiment fit: 0.70
- readability: 0.80
- flags: ['grounding', 'grounding_score below strict threshold', 'sentiment_audit indicates weak tone cues']
- revision attempts: 3
- fallback reached: True

Interpretation
- The stricter mode reduced unsupported bridge language, but it also pushed the run into repeated revisions and eventually a fallback-safe final state.
- That behavior is useful for the UI because it creates an honest warning surface instead of silently pretending the strict run fully succeeded.


## 8. UI evidence and expected execution states

### Idle workspace
![](../assets/screenshots/streamlit_idle.png)

### Generation in progress
![](../assets/screenshots/streamlit_generating.png)

### Successful generation result
![](../assets/screenshots/streamlit_success.png)

### Low-confidence result
![](../assets/screenshots/streamlit_low_confidence.png)

### Blocked / missing-input state
![](../assets/screenshots/streamlit_blocked.png)


## 9. Result logs and execution evidence

The notebook keeps both a provider-backed run and a deterministic fallback run visible. This makes it easy to compare normal output quality with the system's conservative behavior when provider access or image evidence is limited.


In [6]:
import json
from pathlib import Path

example_dir = Path('..') / 'assets' / 'example_runs'
provider = json.loads((example_dir / 'provider_heartwarming_dog_story.json').read_text(encoding='utf-8'))
fallback = json.loads((example_dir / 'fallback_mysterious_ambiguous_story.json').read_text(encoding='utf-8'))

print(provider['story_text'])
print(provider['evaluation_report']['summary'])
print(fallback['story_text'])
print(fallback['evaluation_report']['summary'])


Provider-backed story excerpt
On a sunny afternoon, a person and their dog strolled through the park, surrounded by trees and a winding path. After their walk, the dog joyfully chased a ball, basking in the warmth of the sun and the thrill of play. With each leap and bound, laughter filled the air, creating memories that would linger long after the day was done. As the sun began to set, they returned home, the dog curling up peacefully by the door, content and tired. In that quiet moment, the bond between them felt stronger than ever, a testament to the joy of simple days spent together.

Provider-backed evaluator summary
The story effectively captures a heartwarming day in the park with a dog, showcasing a strong bond between the characters. The grounding is solid, with clear connections to the images, though some ambiguity regarding the dog's breed affects the overall sentiment fit. Sentiment audit: The draft expresses heartwarming clearly through tender, comforting, tender wording.


## 10. Reflection and key takeaways

### What worked well
- The modular `core -> services -> prompts -> infra` split made it straightforward to add a second UI surface without moving business logic into the interface.
- The explicit intermediate contracts made the notebook and README much easier to explain because each stage already had a clear payload.
- Confidence-aware rendering is especially useful for ambiguous or fallback runs because the app can stay transparent instead of silently overselling weak evidence.

### What remains limited
- The public Streamlit deployment surface is prepared, but creating a permanent `streamlit.app` URL still needs the repository owner to complete the Community Cloud sign-in and app creation flow.
- Provider-backed story quality still depends on both image clarity and model variance.
- The Streamlit deployment focuses on the main storytelling path; the richer analysis-correction workflow remains most complete in the Gradio submission app.
